In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [12]:
df = pd.read_csv('/content/ecommerce_demand_forecasting_100k.csv')


In [13]:
df.head(2)

,price,discount_percent,ad_spend,page_views,cart_additions,avg_session_time,competitor_price,seasonality_index,day_of_week,units_sold
0,874.172214,29.038952,14846.810383,8274.171848,1121.678989,200.492771,787.112955,1.154888,2.0,3096.404173
1,1911.285752,26.348582,23475.152935,5227.683915,319.112177,320.413564,1798.080390,1.148193,2.0,672.964327


In [14]:
df.shape

(87389, 10)

In [15]:
features_with_outliers = ['price', 'page_views', 'cart_additions', 'avg_session_time', 'competitor_price', 'units_sold']
df_no_outlier = df.copy()

for col in features_with_outliers:
    Q1 = df_no_outlier[col].quantile(0.25)
    Q3 = df_no_outlier[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    df_no_outlier = df_no_outlier[
        (df_no_outlier[col] >= lower_bound) &
        (df_no_outlier[col] <= upper_bound)
    ]

print(df.shape)
print(df_no_outlier.shape)


(87389, 10)
(85812, 10)


In [ ]:
# Data Preprocessing

In [16]:
x = df.drop(columns='units_sold')
y = df['units_sold']

In [17]:
x_train , x_test , y_train , y_test = train_test_split(x,y,test_size=0.2 , random_state=42)

In [18]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)


In [19]:
df.shape

(87389, 10)

In [ ]:
# ANN Model

In [20]:
model = Sequential()

input_layer = Dense(9, input_shape=(9,))     # yha only feature number dete he

hidden_layer1 = Dense(16, activation='relu')
hidden_layer2 = Dense(32, activation='relu')
hidden_layer3 = Dense(64, activation='relu')

output_layer = Dense(1, activation='linear')  # yha output ki category

model.add(input_layer)
model.add(hidden_layer1)
model.add(hidden_layer2)
model.add(hidden_layer3)
model.add(output_layer)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
# Model Compile

In [21]:
model.compile(loss='mse', optimizer='AdamW', metrics=['mae','mse', tf.keras.metrics.R2Score()])


In [ ]:
# Model Train

In [22]:
early_stop = EarlyStopping(monitor="loss", patience=5)


In [23]:
history = model.fit(
    x_train, y_train,
    batch_size=32,
    epochs=100,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)


Epoch 1/100
1748/1748 ━━━━━━━━━━━━━━━━━━━━ 12s 5ms/step - loss: 11994111.0000 - mae: 2105.5532 - mse: 11994111.0000 - r2_score: -0.0396 - val_loss: 147451.5469 - val_mae: 292.7075 - val_mse: 147451.5469 - val_r2_score: 0.9866
Epoch 2/100
1748/1748 ━━━━━━━━━━━━━━━━━━━━ 16s 4ms/step - loss: 138557.3594 - mae: 283.7504 - mse: 138557.3594 - r2_score: 0.9877 - val_loss: 106050.8672 - val_mae: 248.0285 - val_mse: 106050.8672 - val_r2_score: 0.9904
Epoch 3/100
1748/1748 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - loss: 97331.8516 - mae: 234.4690 - mse: 97331.8516 - r2_score: 0.9913 - val_loss: 70710.0625 - val_mae: 205.3867 - val_mse: 70710.0625 - val_r2_score: 0.9936
Epoch 4/100
1748/1748 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - loss: 67851.2969 - mae: 197.3224 - mse: 67851.2969 - r2_score: 0.9940 - val_loss: 56393.0703 - val_mae: 178.1235 - val_mse: 56393.0703 - val_r2_score: 0.9949
Epoch 5/100
1748/1748 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - loss: 57860.8555 - mae: 181.6616 - mse: 57860.8555 - r2_score: 0.99

## Evaluate You Model

In [29]:
from sklearn.metrics import r2_score
import numpy as np

y_pred = model.predict(x_test).reshape(-1)
y_test = np.array(y_test).reshape(-1)

mask = ~np.isnan(y_test) & ~np.isnan(y_pred)

r2 = r2_score(y_test[mask], y_pred[mask])
print("R2 Score:", r2)


547/547 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
R2 Score: 0.9990142929367164


In [ ]:
# Make Prediction

In [35]:
x_test[1]

array([-0.77385231,  0.42253332, -0.96759532, -1.35162398, -0.79274307,
       -1.56420599, -0.68396394,  1.21976615,  1.00417469])

In [36]:
y_test[1]

np.float64(2401.9178203541687)

In [37]:
arr = np.array([[-0.77385231,  0.42253332, -0.96759532, -1.35162398, -0.79274307, -1.56420599, -0.68396394,  1.21976615,  1.00417469]])
result = model.predict(arr)
result

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step


array([[2413.2007]], dtype=float32)

In [ ]:
model.save("/content/demand_forecast_v1.keras")


In [ ]:
import joblib
joblib.dump(scaler, "/content/feature_scaler.joblib")


['/content/feature_scaler.joblib']

In [ ]:
from sklearn.preprocessing import StandardScaler
import joblib

scaler = StandardScaler()
scaler.fit(X_train)

joblib.dump(scaler, "feature_scaler.joblib")


NameError: name 'X_train' is not defined